In [ ]:
#!/usr/bin/env python3
import os
import random
import pickle
import argparse
import time
import json
import gc
import traceback

import numpy as np
import pandas as pd

from pathlib import Path
from typing import Tuple

try:
    import psutil
except Exception:
    psutil = None

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

import lightgbm as lgb

SEED = 12345
random.seed(SEED)
np.random.seed(SEED)

METRICS_LOG = "farm_metrics.jsonl"
CONSOLIDATED_BEST = "farm_best_clis_all.txt"
CONSOLIDATED_ALT = "farm_alt_clis_all.txt"
CONSOLIDATED_PF = "farm_configs.npy"
MAX_CHUNK = 8

if os.path.exists(METRICS_LOG):
    os.remove(METRICS_LOG)

def append_metrics(row: dict):
    with open(METRICS_LOG, "a") as f:
        f.write(json.dumps(row, default=lambda x: None) + "\n")

PIXEL_SMALL_FREQS  = np.array([300000,574000,738000,930000,1098000,1197000,1328000,1401000,1598000,1704000,1803000], dtype=int)
PIXEL_MEDIUM_FREQS = np.array([400000,553000,696000,799000,910000,1024000,1197000,1328000,1491000,1663000,1836000,1999000,2130000,2253000], dtype=int)
PIXEL_LARGE_FREQS  = np.array([500000,851000,984000,1106000,1277000,1426000,1582000,1745000,1826000,2048000,2188000,2252000,2401000,2507000,2630000,2704000,2802000], dtype=int)

def snap_to_allowed(freq_hz: float, allowed: np.ndarray) -> int:
    arr = np.asarray(allowed, dtype=int)
    idx = int(np.argmin(np.abs(arr - int(round(freq_hz)))))
    return int(arr[idx])

df_raw = pd.read_excel("mobile_farm_data.xlsx").copy()
df = df_raw.dropna(subset=["energy","time"]).copy()

if "chunk_size" in df:
    df["chunk_size"] = pd.to_numeric(df["chunk_size"], errors="coerce").fillna(1).astype(int)
    df["chunk_size"] = df["chunk_size"].clip(upper=MAX_CHUNK)
else:
    df["chunk_size"] = 1

max_workers = int(df["number_of_workers"].max()) if "number_of_workers" in df else 4
max_workers = max(max_workers, 1)
for i in range(1, max_workers + 1):
    wcol = f"workload_size_w{i}"
    tcol = f"w{i}_threads"
    if wcol not in df:
        df[wcol] = 0
    if tcol not in df:
        df[tcol] = 0

for c in ("small_cores_on","medium_cores_on","large_cores_on"):
    if c not in df:
        df[c] = 0

cpu_cols = [c for c in df.columns if c.startswith("cpu") and c.endswith("_freq")]
small_cpu_cols = [f"cpu{i}_freq" for i in range(0,4) if f"cpu{i}_freq" in df.columns]
medium_cpu_cols= [f"cpu{i}_freq" for i in range(4,6) if f"cpu{i}_freq" in df.columns]
large_cpu_cols = [f"cpu{i}_freq" for i in range(6,8) if f"cpu{i}_freq" in df.columns]
all_cpu_freq_cols = cpu_cols
if (not small_cpu_cols) and all_cpu_freq_cols:
    n = len(all_cpu_freq_cols)
    s1 = max(1, n//3); s2 = max(s1+1, 2*n//3)
    small_cpu_cols = all_cpu_freq_cols[:s1]; medium_cpu_cols = all_cpu_freq_cols[s1:s2]; large_cpu_cols = all_cpu_freq_cols[s2:]

if small_cpu_cols:
    df["avg_small_freq"]  = df[small_cpu_cols].mean(axis=1)
    df["min_small_freq"]  = df[small_cpu_cols].min(axis=1)
    df["med_small_freq"]  = df[small_cpu_cols].median(axis=1)
    df["max_small_freq"]  = df[small_cpu_cols].max(axis=1)
else:
    df["avg_small_freq"] = float(np.mean(PIXEL_SMALL_FREQS))
    df["min_small_freq"] = float(np.min(PIXEL_SMALL_FREQS))
    df["med_small_freq"] = float(np.median(PIXEL_SMALL_FREQS))
    df["max_small_freq"] = float(np.max(PIXEL_SMALL_FREQS))

if medium_cpu_cols:
    df["avg_medium_freq"] = df[medium_cpu_cols].mean(axis=1)
    df["min_medium_freq"] = df[medium_cpu_cols].min(axis=1)
    df["med_medium_freq"] = df[medium_cpu_cols].median(axis=1)
    df["max_medium_freq"] = df[medium_cpu_cols].max(axis=1)
else:
    df["avg_medium_freq"] = float(np.mean(PIXEL_MEDIUM_FREQS))
    df["min_medium_freq"] = float(np.min(PIXEL_MEDIUM_FREQS))
    df["med_medium_freq"] = float(np.median(PIXEL_MEDIUM_FREQS))
    df["max_medium_freq"] = float(np.max(PIXEL_MEDIUM_FREQS))

if large_cpu_cols:
    df["avg_large_freq"]  = df[large_cpu_cols].mean(axis=1)
    df["min_large_freq"]  = df[large_cpu_cols].min(axis=1)
    df["med_large_freq"]  = df[large_cpu_cols].median(axis=1)
    df["max_large_freq"]  = df[large_cpu_cols].max(axis=1)
else:
    df["avg_large_freq"] = float(np.mean(PIXEL_LARGE_FREQS))
    df["min_large_freq"] = float(np.min(PIXEL_LARGE_FREQS))
    df["med_large_freq"] = float(np.median(PIXEL_LARGE_FREQS))
    df["max_large_freq"] = float(np.max(PIXEL_LARGE_FREQS))

df["total_cores_on"] = df["small_cores_on"] + df["medium_cores_on"] + df["large_cores_on"]
df["imbalance_sm"] = abs(df["small_cores_on"] - df["medium_cores_on"])
df["imbalance_sl"] = abs(df["small_cores_on"] - df["large_cores_on"])
df["imbalance_ml"] = abs(df["medium_cores_on"] - df["large_cores_on"])
df["ratio_small_to_medium"] = df["small_cores_on"] / (df["medium_cores_on"] + 1e-6)
df["ratio_small_to_large"]  = df["small_cores_on"] / (df["large_cores_on"] + 1e-6)
df["ratio_medium_to_large"] = df["medium_cores_on"] / (df["large_cores_on"] + 1e-6)
df["total_processing_threads"] = sum(df.get(f"w{i}_threads", 0) for i in range(1, max_workers + 1))

cat_cols = [c for c in ["workload_type","core_binding_type","workload_balance"] if c in df]
num_cols = [
    "number_of_workers","chunk_size","total_processing_threads",
    "small_cores_on","medium_cores_on","large_cores_on","total_cores_on",
    "imbalance_sm","imbalance_sl","imbalance_ml",
    "ratio_small_to_medium","ratio_small_to_large","ratio_medium_to_large",
    "avg_small_freq","med_small_freq","min_small_freq","max_small_freq",
    "avg_medium_freq","med_medium_freq","min_medium_freq","max_medium_freq",
    "avg_large_freq","med_large_freq","min_large_freq","max_large_freq"
] + [f"workload_size_w{i}" for i in range(1, max_workers+1)] + [f"w{i}_threads" for i in range(1, max_workers+1)]
num_cols = [c for c in num_cols if c in df]

X = pd.concat([df[cat_cols].astype(str), df[num_cols]], axis=1)

y_raw = df[["energy","time"]].copy()
y = y_raw.copy()
y["time"] = np.log1p(y_raw["time"])

preproc = ColumnTransformer([
    ("num", SimpleImputer(strategy="mean"), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse=False), cat_cols)
])

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.15, random_state=SEED)

lgb_params_energy = {"n_estimators": 200, "random_state": SEED, "n_jobs": -1}
lgb_params_time   = {"n_estimators": 300, "random_state": SEED, "n_jobs": -1, "objective": "huber"}

model_energy = Pipeline([
    ("prep", preproc),
    ("reg", lgb.LGBMRegressor(**lgb_params_energy))
])

model_time = Pipeline([
    ("prep", preproc),
    ("reg", lgb.LGBMRegressor(**lgb_params_time))
])

def fit_with_metrics_single(pipe: Pipeline, X_train: pd.DataFrame, y_train: pd.DataFrame) -> Tuple[Pipeline, float, int]:
    start = time.perf_counter()
    peak_mem = -1
    if psutil is None:
        pipe.fit(X_train, y_train)
        return pipe, time.perf_counter() - start, -1
    proc = psutil.Process()
    import threading
    done = threading.Event()
    mem_samples = []
    def sampler():
        while not done.is_set():
            try:
                mem_samples.append(proc.memory_info().rss)
            except Exception:
                pass
            time.sleep(0.01)
    thread = threading.Thread(target=sampler, daemon=True)
    thread.start()
    try:
        pipe.fit(X_train, y_train)
    finally:
        done.set()
        thread.join(timeout=1.0)
    train_time = time.perf_counter() - start
    peak_mem = int(max(mem_samples) if mem_samples else -1)
    return pipe, train_time, peak_mem

model_energy, train_time_e, peak_mem_bytes_e = fit_with_metrics_single(model_energy, Xtr, ytr["energy"])
model_time, train_time_t, peak_mem_bytes_t = fit_with_metrics_single(model_time, Xtr, ytr["time"])

train_time_s = float(train_time_e + train_time_t)
peak_mem_bytes = int(max(peak_mem_bytes_e, peak_mem_bytes_t))

pred_e_te = model_energy.predict(Xte)
pred_t_te_log = model_time.predict(Xte)
pred_t_te = np.expm1(pred_t_te_log)

mae_e = mean_absolute_error(yte["energy"], pred_e_te)
mae_t = mean_absolute_error(np.expm1(yte["time"]), pred_t_te)
r2_e  = r2_score(yte["energy"], pred_e_te)
r2_t  = r2_score(np.expm1(yte["time"]), pred_t_te) if np.isfinite(np.expm1(yte["time"]).std()) else float("nan")

print(f"Surrogate trained → MAE energy={mae_e:.3f}, MAE time={mae_t:.3f}")
print(f"R2            → energy={r2_e:.3f}, time={r2_t:.3f}")
print(f"Train time (s): {train_time_s:.3f}, Peak mem (bytes): {peak_mem_bytes}")

baseline_mae_e = float(np.mean(np.abs((yte["energy"] - yte["energy"].mean()))))
baseline_mae_t = float(np.mean(np.abs(np.expm1(yte["time"]) - np.expm1(yte["time"]).mean())))
yte_std_energy = float(yte["energy"].std()); yte_std_time = float(np.expm1(yte["time"]).std())
r2_reliable = (yte_std_energy > 1e-6) and (yte_std_time > 1e-6)

tmp_model_file = "farm_model_tmp_mobile.pkl"
with open(tmp_model_file, "wb") as f:
    pickle.dump({"energy": model_energy, "time": model_time}, f, protocol=pickle.HIGHEST_PROTOCOL)
model_size_bytes = os.path.getsize(tmp_model_file)
os.remove(tmp_model_file)
print(f"Model serialized size (bytes): {model_size_bytes}")

def predict_perf(pipe: Pipeline, Xc: pd.DataFrame, reps: int = 5) -> Tuple[float, float, np.ndarray]:
    _ = pipe.predict(Xc.iloc[:min(10, len(Xc))])
    times = []
    preds_local = None
    for _ in range(reps):
        t0 = time.perf_counter()
        preds_local = pipe.predict(Xc)
        t1 = time.perf_counter()
        times.append((t1 - t0) / max(1, len(Xc)))
    return float(np.median(times)), float(np.std(times)), preds_local

with open("farm_columns.txt", "w") as f:
    for col in cat_cols + num_cols:
        f.write(col + "\n")

with open("farm_model.pkl", "wb") as f:
    pickle.dump({"energy": model_energy, "time": model_time, "preproc_cols": {"cat": cat_cols, "num": num_cols}}, f, protocol=pickle.HIGHEST_PROTOCOL)

def lgb_tree_to_dict(tree_struct):
    if "leaf_value" in tree_struct:
        return {"leaf": True, "value": float(tree_struct["leaf_value"])}
    feature = int(tree_struct.get("split_feature", -1))
    threshold = float(tree_struct.get("threshold", 0.0))
    left = lgb_tree_to_dict(tree_struct["left_child"])
    right = lgb_tree_to_dict(tree_struct["right_child"])
    return {"leaf": False, "feature": feature, "threshold": threshold, "left": left, "right": right}

def lgb_model_to_forest(estimator):
    booster = estimator.booster_
    dump = booster.dump_model()
    trees = []
    for t in dump.get("tree_info", []):
        tree_struct = t.get("tree_structure", {})
        trees.append(lgb_tree_to_dict(tree_struct))
    return trees

def extract_forests(pipe: Pipeline):
    reg = pipe.named_steps["reg"]
    booster = reg.booster_
    dump = booster.dump_model()
    trees = []
    for t in dump.get("tree_info", []):
        trees.append(lgb_tree_to_dict(t.get("tree_structure", {})))
    return trees

forests_energy = extract_forests(model_energy)
forests_time = extract_forests(model_time)

spec = {
    "cat_cols": cat_cols,
    "num_cols": num_cols,
    "forests_energy": forests_energy,
    "forests_time": forests_time
}

with open("farm_model_numpy.pkl", "wb") as f:
    pickle.dump(spec, f, protocol=pickle.HIGHEST_PROTOCOL)

print("✅ Saved farm_model.pkl, farm_model_numpy.pkl and farm_columns.txt (LightGBM Mobile)")

def pareto_mask(E: np.ndarray, T: np.ndarray) -> np.ndarray:
    keep = np.ones(len(E), dtype=bool)
    for i in range(len(E)):
        if not keep[i]:
            continue
        dom = (E <= E[i]) & (T <= T[i]) & ((E < E[i]) | (T < T[i]))
        dom[i] = False
        if dom.any():
            keep[i] = False
    return keep

def random_candidates(workload_type: str, wl: list, number_of_workers: int, n_candidates: int = 2000):
    wl_arr = np.atleast_1d(wl)
    if wl_arr.size == 1:
        wl_arr = np.full(number_of_workers, wl_arr.item(), dtype=int)

    mask = (df["workload_type"] == workload_type) & (pd.to_numeric(df.get("number_of_workers"), errors="coerce").fillna(-1).astype(int) == number_of_workers)
    for i in range(1, number_of_workers + 1):
        col = f"workload_size_w{i}"
        ser = pd.to_numeric(df.get(col, pd.Series(dtype=float)), errors="coerce").fillna(-1).astype(int)
        mask &= (ser == wl_arr[i-1])
    real_rows = df[mask]
    if real_rows.shape[0] > 0:
        rows = []
        keys = list(real_rows.columns)
        for _, r in real_rows.iterrows():
            row = {k: r[k] for k in keys}
            rows.append(row)
        return pd.DataFrame(rows), int(real_rows.shape[0])

    try:
        core_combos = [(int(s), int(m), int(l))
                       for s in np.unique(pd.to_numeric(df.get("small_cores_on"), errors="coerce").dropna())
                       for m in np.unique(pd.to_numeric(df.get("medium_cores_on"), errors="coerce").dropna())
                       for l in np.unique(pd.to_numeric(df.get("large_cores_on"), errors="coerce").dropna())
                       if (int(s) + int(m) + int(l)) > 0]
    except Exception:
        core_combos = [(1, 0, 0)]

    if not core_combos:
        core_combos = [(1, 0, 0)]

    small_freq_domain = np.unique(pd.to_numeric(df.get("avg_small_freq"), errors="coerce").dropna().astype(int)) if "avg_small_freq" in df else PIXEL_SMALL_FREQS
    medium_freq_domain = np.unique(pd.to_numeric(df.get("avg_medium_freq"), errors="coerce").dropna().astype(int)) if "avg_medium_freq" in df else PIXEL_MEDIUM_FREQS
    large_freq_domain = np.unique(pd.to_numeric(df.get("avg_large_freq"), errors="coerce").dropna().astype(int)) if "avg_large_freq" in df else PIXEL_LARGE_FREQS

    if "total_processing_threads" in df:
        t_choices = pd.to_numeric(df["total_processing_threads"], errors="coerce").dropna().unique()
        t_choices = [int(x) for x in t_choices if not np.isnan(x)]
    else:
        t_choices = []

    chunk_choices = []
    if "chunk_size" in df:
        try:
            chunk_choices = list(np.unique(pd.to_numeric(df["chunk_size"], errors="coerce").dropna().astype(int)))
        except Exception:
            chunk_choices = []
    if not chunk_choices:
        chunk_choices = [1]
    chunk_choices = [int(min(MAX_CHUNK, c)) for c in chunk_choices if c > 0]
    if not chunk_choices:
        chunk_choices = [1]

    wb_domain = list(df.get("workload_balance", pd.Series(dtype=str)).astype(str).unique()) if "workload_balance" in df else ["unknown"]
    if not wb_domain:
        wb_domain = ["unknown"]

    rows = []
    for _ in range(n_candidates):
        s_on, m_on, l_on = random.choice(core_combos)
        fs_s = np.random.choice(small_freq_domain, size=min(4, len(small_freq_domain)), replace=True)
        fs_m = np.random.choice(medium_freq_domain, size=min(4, len(medium_freq_domain)), replace=True)
        fs_l = np.random.choice(large_freq_domain, size=min(4, len(large_freq_domain)), replace=True)
        avg_s = float(np.mean(fs_s)) if len(fs_s) else float(np.mean(PIXEL_SMALL_FREQS))
        avg_m = float(np.mean(fs_m)) if len(fs_m) else float(np.mean(PIXEL_MEDIUM_FREQS))
        avg_l = float(np.mean(fs_l)) if len(fs_l) else float(np.mean(PIXEL_LARGE_FREQS))

        if t_choices:
            total_threads = int(random.choice(t_choices))
        else:
            total_threads = random.randint(number_of_workers, max(number_of_workers, number_of_workers * 4))

        base = [1] * number_of_workers
        leftover = max(0, total_threads - number_of_workers)
        if leftover > 0:
            extra = list(np.random.multinomial(leftover, [1/number_of_workers] * number_of_workers))
        else:
            extra = [0] * number_of_workers
        alloc = [int(a + b) for a, b in zip(base, extra)]

        chunk = int(random.choice(chunk_choices))
        chunk = min(chunk, MAX_CHUNK)
        wb = random.choice(wb_domain)

        entry = {
            "workload_type": workload_type,
            "number_of_workers": number_of_workers,
            "workload_balance": wb,
            "chunk_size": chunk,
            "core_binding_type": "manual_roundrobin",
            "small_cores_on": int(s_on), "medium_cores_on": int(m_on), "large_cores_on": int(l_on),
            "total_cores_on": int(s_on + m_on + l_on),
            "imbalance_sm": abs(int(s_on) - int(m_on)), "imbalance_sl": abs(int(s_on) - int(l_on)), "imbalance_ml": abs(int(m_on) - int(l_on)),
            "ratio_small_to_medium": float(s_on / (m_on + 1e-6)),
            "ratio_small_to_large": float(s_on / (l_on + 1e-6)),
            "ratio_medium_to_large": float(m_on / (l_on + 1e-6)),
            "avg_small_freq": avg_s, "med_small_freq": float(np.median(fs_s)) if len(fs_s) else avg_s,
            "min_small_freq": float(np.min(fs_s)) if len(fs_s) else avg_s, "max_small_freq": float(np.max(fs_s)) if len(fs_s) else avg_s,
            "avg_medium_freq": avg_m, "med_medium_freq": float(np.median(fs_m)) if len(fs_m) else avg_m,
            "min_medium_freq": float(np.min(fs_m)) if len(fs_m) else avg_m, "max_medium_freq": float(np.max(fs_m)) if len(fs_m) else avg_m,
            "avg_large_freq": avg_l, "med_large_freq": float(np.median(fs_l)) if len(fs_l) else avg_l,
            "min_large_freq": float(np.min(fs_l)) if len(fs_l) else avg_l, "max_large_freq": float(np.max(fs_l)) if len(fs_l) else avg_l,
            "total_processing_threads": int(total_threads)
        }

        for i in range(1, number_of_workers + 1):
            entry[f"workload_size_w{i}"] = int(wl_arr[i-1])
            entry[f"w{i}_threads"] = int(alloc[i-1])

        for i in range(number_of_workers + 1, max_workers + 1):
            entry[f"workload_size_w{i}"] = 0
            entry[f"w{i}_threads"] = 0

        rows.append(entry)

    return pd.DataFrame(rows), 0

def build_cli(row: dict) -> str:
    s_on = int(row.get("small_cores_on", 0)); m_on = int(row.get("medium_cores_on", 0)); l_on = int(row.get("large_cores_on", 0))
    sf_raw = float(row.get("avg_small_freq", float(np.mean(PIXEL_SMALL_FREQS))))
    mf_raw = float(row.get("avg_medium_freq", float(np.mean(PIXEL_MEDIUM_FREQS))))
    lf_raw = float(row.get("avg_large_freq", float(np.mean(PIXEL_LARGE_FREQS))))

    sf_hz = snap_to_allowed(sf_raw, PIXEL_SMALL_FREQS)
    mf_hz = snap_to_allowed(mf_raw, PIXEL_MEDIUM_FREQS)
    lf_hz = snap_to_allowed(lf_raw, PIXEL_LARGE_FREQS)

    sf_ghz = sf_hz / 1e6; mf_ghz = mf_hz / 1e6; lf_ghz = lf_hz / 1e6

    workers = int(row.get("number_of_workers", 1))
    wt = row.get("workload_type", "unknown")
    chunk = int(row.get("chunk_size", 1))
    if chunk > MAX_CHUNK:
        chunk = MAX_CHUNK
        print(f"⚠️ chunk_size clipped to {MAX_CHUNK} for CLI (original > {MAX_CHUNK})")
    sizes = " ".join(str(int(row.get(f"workload_size_w{i}", 0))) for i in range(1, workers+1))
    thrs = " ".join(str(int(row.get(f"w{i}_threads", 1))) for i in range(1, workers+1))

    freq_tokens = f"--small-freq {sf_ghz:.3f}GHz --medium-freq {mf_ghz:.3f}GHz --large-freq {lf_ghz:.3f}GHz "

    return (f"--small-cores {s_on} --medium-cores {m_on} --large-cores {l_on} "
            f"{freq_tokens}"
            f"./farm {workers} {wt} {sizes} {chunk} {thrs}")

def recommend_all(workload_type: str, wl: list, number_of_workers: int, n_candidates: int = 2000, K: int = 4):
    cand_df, real_count = random_candidates(workload_type, wl, number_of_workers, n_candidates)
    if real_count == 0:
        print("⚠️ No exact‑match real runs; using ML for all candidates")

    Xc = pd.concat([cand_df[cat_cols].astype(str), cand_df[num_cols]], axis=1)
    pred_latency_cand_s_e, pred_latency_cand_std_s_e, preds_e = predict_perf(model_energy, Xc, reps=3)
    pred_latency_cand_s_t, pred_latency_cand_std_s_t, preds_t_log = predict_perf(model_time, Xc, reps=3)

    preds_time = np.expm1(preds_t_log)
    cand_df["pred_energy_raw"] = preds_e
    cand_df["pred_time_raw"]   = preds_time

    calib = Path("farm_calib.csv")
    if calib.exists():
        df_cal = pd.read_csv(calib)
        Xcal = pd.concat([df_cal[cat_cols].astype(str), df_cal[num_cols]], axis=1)
        pcal_e = model_energy.predict(Xcal)
        pcal_t_log = model_time.predict(Xcal)
        pcal_t = np.expm1(pcal_t_log)

        lr_e = LinearRegression().fit(pcal_e.reshape(-1,1), df_cal["energy"].values)
        lr_t = LinearRegression().fit(pcal_t.reshape(-1,1), df_cal["time"].values)

        cand_df["energy"] = lr_e.predict(cand_df["pred_energy_raw"].values.reshape(-1,1))
        cand_df["time"]   = lr_t.predict(cand_df["pred_time_raw"].values.reshape(-1,1))
        print("✅ Applied linear calibration (measured-space) to energy and time")
    else:
        cand_df["energy"] = cand_df["pred_energy_raw"]
        cand_df["time"]   = cand_df["pred_time_raw"]
        print("⚠️ farm_calib.csv not found; skipping calibration")

    mask = pareto_mask(cand_df["energy"].to_numpy(), cand_df["time"].to_numpy())
    pf = cand_df[mask].copy().reset_index(drop=True)

    pf["score_balanced"] = pf.energy / (pf.energy.max() if pf.energy.max() else 1.0) + pf.time / (pf.time.max() if pf.time.max() else 1.0)
    pf = pf.sort_values(by=["energy", "time"]).reset_index(drop=True)

    pf_df = pd.concat([pf[cat_cols].astype(str), pf[num_cols]], axis=1).reset_index(drop=True)
    pf_df["_sort_key_e"] = pf["energy"].astype(float)
    pf_df["_sort_key_t"] = pf["time"].astype(float)
    pf_df = pf_df.sort_values(by=["_sort_key_e", "_sort_key_t"]).reset_index(drop=True)
    pf_df = pf_df.drop(columns=["_sort_key_e", "_sort_key_t"])
    pf_arr = pf_df.to_numpy(dtype=object)

    np.save("farm_configs.npy", pf_arr, allow_pickle=True)
    np.savez_compressed("farm_configs.npz", X=pf_arr)
    with open("farm_columns.txt", "w") as f:
        for col in cat_cols + num_cols:
            f.write(col + "\n")
    print("✅ Saved farm_configs.npy, farm_configs.npz and farm_columns.txt")

    def top_alts(df_local, key):
        df2 = df_local.copy()
        best_idx = df2[key].idxmin()
        alts = df2.nsmallest(K+1, key).drop(best_idx, errors="ignore")
        seen, uniq = set(), []
        for _, r in alts.iterrows():
            cli = build_cli(r.to_dict())
            if cli in seen: continue
            seen.add(cli); uniq.append(cli)
            if len(uniq) == K: break
        return uniq

    results = {
        "energy": build_cli(pf.nsmallest(1, "energy").iloc[0].to_dict()) if not pf.empty else "",
        "time": build_cli(pf.nsmallest(1, "time").iloc[0].to_dict()) if not pf.empty else "",
        "balanced": build_cli(pf.nsmallest(1, "score_balanced").iloc[0].to_dict()) if not pf.empty else ""
    }

    alts = {"energy": top_alts(pf, "energy"), "time": top_alts(pf, "time"), "balanced": top_alts(pf, "score_balanced")}
    pf_keys = set(build_cli(r.to_dict()) for _, r in pf.iterrows())
    topk_energy = set(alts["energy"])
    topk_time   = set(alts["time"])
    topk_bal    = set(alts["balanced"])

    metrics_row = {
        "timestamp": time.time(), "seed": SEED, "workload_type": workload_type, "workload_sizes": list(map(int, wl)),
        "number_of_workers": number_of_workers, "n_candidates": len(cand_df),
        "train_time_s": float(train_time_s), "peak_mem_bytes": int(peak_mem_bytes), "model_size_bytes": int(model_size_bytes),
        "pred_latency_per_sample_ms_candidates_energy": float(pred_latency_cand_s_e*1e3),
        "pred_latency_std_ms_candidates_energy": float(pred_latency_cand_std_s_e*1e3),
        "pred_latency_per_sample_ms_candidates_time": float(pred_latency_cand_s_t*1e3),
        "pred_latency_std_ms_candidates_time": float(pred_latency_cand_std_s_t*1e3),
        "pareto_count": int(len(pf_keys)),
        "best_energy_cli": results["energy"], "best_time_cli": results["time"], "best_balanced_cli": results["balanced"],
        "topk_energy": sorted(list(topk_energy)), "topk_time": sorted(list(topk_time)), "topk_balanced": sorted(list(topk_bal)),
        "baseline_mae_energy": baseline_mae_e, "baseline_mae_time": baseline_mae_t,
        "yte_std_energy": yte_std_energy, "yte_std_time": yte_std_time, "r2_reliable": bool(r2_reliable),
        "max_chunk": MAX_CHUNK
    }
    append_metrics(metrics_row)
    artifacts = {"pf_keys": pf_keys, "best": {"energy": results["energy"], "time": results["time"], "balanced": results["balanced"]},
                 "topk": {"energy": topk_energy, "time": topk_time, "balanced": topk_bal}}
    return results, alts, artifacts

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Train LightGBM farm surrogate & emit Pareto configs")
    parser.add_argument("--n_candidates", "-c", type=int, default=2000)
    parser.add_argument("--K", "-k", type=int, default=4, help="Max number of alternatives per objective (3-4 recommended)")
    args, _ = parser.parse_known_args()

    wts = ["cpu_only","memory_only","cpu_and_memory","memory_and_cpu"]
    vectors = {
        "cpu_only":         [[1_000_000]*3, [5_000_000]*3, [10_000_000]*3, [1_000_000,2_000_000,3_000_000]],
        "memory_only":      [[2_000_000]*4, [4_000_000]*4],
        "cpu_and_memory":   [[8_000_000]*2, [16_000_000]*2],
        "memory_and_cpu":   [[8_000_000]*2, [16_000_000]*2],
    }

    all_pf = []
    mae_energy_list = []; mae_time_list = []
    r2_energy_list = []; r2_time_list = []
    consolidated_best_clis = []; consolidated_alt_clis = set()
    combos_processed = 0

    for wt in wts:
        for wl in vectors.get(wt, []):
            num_workers = len(wl)
            results, alts, artifacts = recommend_all(workload_type=wt, wl=wl, number_of_workers=num_workers,
                                                    n_candidates=args.n_candidates, K=args.K)
            vec_str = "_".join(str(x) for x in wl)
            combo = f"{wt}_{num_workers}workers_{vec_str}"
            print(f"\n=== RESULTS for {combo} ===")
            print("ENERGY   →", results["energy"])
            print("TIME     →", results["time"])
            print("BALANCED →", results["balanced"])
            print(f"\n--- ALTERNATIVES for {combo} ---")
            for obj in ("balanced","energy","time"):
                for cli in alts[obj]:
                    print(" ", cli)
            try:
                pf_arr = np.load("farm_configs.npy", allow_pickle=True)
            except Exception:
                pf_arr = np.zeros((0, len(cat_cols)+len(num_cols)), dtype=object)
            all_pf.append(pf_arr)
            consolidated_best_clis.append({"combo": combo, "best_energy": artifacts["best"]["energy"],
                                           "best_time": artifacts["best"]["time"], "best_balanced": artifacts["best"]["balanced"]})
            for s in artifacts["pf_keys"]:
                consolidated_alt_clis.add(s)
            for objset in artifacts["topk"].values():
                for k in objset:
                    consolidated_alt_clis.add(k)
            mae_energy_list.append(float(mae_e) if 'mae_e' in globals() else None)
            mae_time_list.append(float(mae_t) if 'mae_t' in globals() else None)
            r2_energy_list.append(float(r2_e) if 'r2_e' in globals() else None)
            r2_time_list.append(float(r2_t) if 'r2_t' in globals() else None)
            combos_processed += 1
            print(f"✅ Processed {combo}")

    combined = np.vstack(all_pf) if all_pf else np.zeros((0, len(cat_cols)+len(num_cols)))
    np.save(CONSOLIDATED_PF, combined)
    with open(CONSOLIDATED_BEST, "w") as f:
        for entry in consolidated_best_clis:
            f.write(json.dumps(entry) + "\n")
    with open(CONSOLIDATED_ALT, "w") as f:
        for cli in sorted(consolidated_alt_clis):
            f.write(cli + "\n")

    def summarize(lst):
        if not lst: return {"mean": None, "std": None, "count": 0}
        a = np.array([x for x in lst if x is not None], dtype=float)
        if a.size == 0: return {"mean": None, "std": None, "count": 0}
        return {"mean": float(a.mean()), "std": float(a.std(ddof=0)), "count": int(a.size)}

    totals = {
        "mae_energy": summarize(mae_energy_list),
        "mae_time": summarize(mae_time_list),
        "r2_energy": summarize(r2_energy_list),
        "r2_time": summarize(r2_time_list),
        "combos_processed": combos_processed
    }

    summary_out = {
        "totals": totals,
        "global_test": {"mae_energy": float(mae_e) if 'mae_e' in globals() else None, "mae_time": float(mae_t) if 'mae_t' in globals() else None,
                        "r2_energy": float(r2_e) if 'r2_e' in globals() else None, "r2_time": float(r2_t) if 'r2_t' in globals() else None},
        "model": {"n_estimators_energy": lgb_params_energy["n_estimators"], "n_estimators_time": lgb_params_time["n_estimators"],
                  "model_size_bytes": model_size_bytes, "train_time_s": train_time_s, "peak_mem_bytes": peak_mem_bytes},
        "notes": "Farm tri-cluster: small/medium/large cores and Pixel-6a allowed freqs. Time trained in log1p-space; calibration applied on measured space if farm_calib.csv present.",
        "policy": {"max_chunk_size": MAX_CHUNK}
    }
    with open("farm_aggregated_summary.json", "w") as f:
        json.dump(summary_out, f, indent=2)

    print(f"\n✅ Saved COMBINED {CONSOLIDATED_PF} ({combined.shape[0]} total configs across all combos)")
    print(f"✅ Wrote consolidated best CLIs → {CONSOLIDATED_BEST}")
    print(f"✅ Wrote consolidated alt CLIs  → {CONSOLIDATED_ALT}")
    print(f"✅ Wrote aggregated summary     → farm_aggregated_summary.json")
    print(f"✅ Per-combo metrics log        → {METRICS_LOG}")
